# Summary Evaluators

### Setup

In [37]:
# You can set them inline
# import os
# os.environ["OPENAI_API_KEY"] = ""
# os.environ["LANGSMITH_API_KEY"] = ""
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-notebook"

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

### Task

我们的任务是分析随机语句的有害性，将其分类为 `Toxic` or `Not toxic`. 

In [2]:
from langsmith import Client

client = Client()
dataset = client.clone_public_dataset(
    "https://smith.langchain.com/public/89ef0d44-a252-4011-8bb8-6a114afc1522/d"
)

This is a simple toxicity classifier!

In [19]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field

openai_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

SUMMARY_HUMAN_PROMPT = """
This is the statement: {statement} 
Output your response in JSON format with the key 'toxicity' value must be an string 
The value for toxicity can only be 'Toxic' or 'Not toxic'.
"""

class Toxicity(BaseModel):
    toxicity: str = Field(description="""'Toxic' if this the statement is toxic, 'Not toxic' if the statement is not toxic.""")

def good_classifier(inputs: dict) -> dict:
    completion = openai_client.beta.chat.completions.parse(
        model="qwen3-max",
        messages=[
            {
                "role": "user",
                # "content": f"This is the statement: {inputs['statement']}"
                "content": SUMMARY_HUMAN_PROMPT.format(statement=inputs['statement'])
            }
        ],
        response_format=Toxicity,
    )

    toxicity_score = completion.choices[0].message.parsed.toxicity
    return {"class": toxicity_score}

### Summary Evaluator

以下是摘要评估函数可访问的字段：
- `inputs: list[dict]`: 数据集中示例的输入列表
- `outputs: list[dict]`: 运行目标程序处理每个输入后生成的dict输出列表
- `reference_outputs: list[dict]`: 数据集中示例的参考输出列表
- `runs: list[Run]`: 运行目标在数据集上运行时生成的Run对象列表
- `examples: list[Example]`: 完整数据集示例列表，包含示例输入、输出（如有）及元数据（如有）。

现在我们来定义摘要评估器！这里我们将计算F1分数，它是精确率和召回率的组合指标。

此类指标只能基于实验中的所有示例进行计算，因此我们的评估器需要接收两个参数：输出列表和参考输出列表。

In [20]:
def f1_score_summary_evaluator(outputs: list[dict], reference_outputs: list[dict]) -> dict:
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    for output_dict, reference_output_dict in zip(outputs, reference_outputs):
        output = output_dict["class"]
        reference_output = reference_output_dict["class"]
        if output == "Toxic" and reference_output == "Toxic":
            true_positives += 1
        elif output == "Toxic" and reference_output == "Not toxic":
            false_positives += 1
        elif output == "Not toxic" and reference_output == "Toxic":
            false_negatives += 1

    if true_positives == 0:
        return {"key": "f1_score", "score": 0.0}

    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    f1_score = 2 * (precision * recall) / (precision + recall)
    return {"key": "f1_score", "score": f1_score}


请注意，我们传递的是 `f1_score_summary_evaluator` 作为摘要评估器！

In [21]:
results = client.evaluate(
    good_classifier,
    data=dataset,
    summary_evaluators=[f1_score_summary_evaluator],
    experiment_prefix="Good classifier"
)

View the evaluation results for experiment: 'Good classifier-4d8137c3' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/e6aa202c-b180-40a6-bd35-39f4fa28bed8/compare?selectedSessions=9bd1d717-9f49-4054-98ea-e883d32316db




9it [00:10,  1.14s/it]
